# Brain Tumor MRI: EDA and Quantum Model Comparison
**Complete Google Colab workflow | four classes | PennyLane + PyTorch**

## Goal
Download the dataset, inspect image quality and leakage, create reproducible splits,
train six quantum variants and matched classical controls, then plot **held-out test
accuracy**, macro F1, learning curves, and confusion matrices. Results are calculated
from your training run; this notebook contains no invented MRI accuracy values.

Dataset: [Masoud Nickparvar — Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset).
Classes: **glioma, meningioma, notumor, pituitary**. Counts are read from downloaded files.

### How to run
1. Upload this `.ipynb` to [Google Colab](https://colab.research.google.com/).
2. Use a fresh Python runtime. A T4 GPU helps the CNNs; all quantum circuits run on CPU.
3. Run Cell 1, choose settings in Cell 3, and run all remaining cells in order.
4. Default `RUN_MODE = "full"` uses every retained image and trains up to 20 epochs.
   Use `"smoke"` for a short pipeline check or `"quick"` for an exploratory subset.
5. If Kaggle download is blocked, set `DATA_SOURCE = "upload"` and upload the ZIP
   downloaded using the dataset page's **Download** button.

| Model | Quantum component | Classifier / purpose |
|---|---|---|
| Quanvolution | Fixed 4-qubit circuit on 2×2 image patches | Train a CNN on cached quantum feature maps |
| Hybrid quantum CNN | CNN → six angles → trainable 6-qubit circuit | Jointly train CNN, circuit, and class readout |
| VQC | 6-qubit circuit with repeated angle encoding | Four class probabilities from two measured qubits |
| QGAN + auxiliary classifier | Conditional 6-qubit quantum generator | Train a latent-space GAN; evaluate the discriminator's tumor-class head |
| Angle QNN | Encode six PCA features once, then train a 6-qubit circuit | Expectation values → linear class readout |
| Amplitude QNN | Encode up to 64 PCA features into six qubits | Expectation values → linear class readout |
| Classical CNN / CNN bottleneck | None | Image controls, including replacement of the hybrid circuit |
| Latent MLP / amplitude-feature MLP | None | Controls with the same inputs as QGAN/angle models and amplitude QNN |
| Patch CNN control | None | The same CNN head and patch resolution as quanvolution |

### What the comparison means
This is a **classically simulated quantum experiment**, using exact expectation values
(`shots=None`) and simulator backpropagation. It does not measure quantum hardware
speed, shot noise, or a quantum advantage. The image resolutions and compression differ
across families, so compare each model with its corresponding classical control.

**Quanvolution is not the hierarchical convolution-and-pooling QCNN architecture**
of Cong et al. Here it means the patch-based quantum convolution requested in the prompt.
The quanvolution circuit is deliberately fixed, following Henderson et al.; its CNN head
is trained. Other discriminative quantum circuits and the QGAN generator are trainable.

**QGAN accuracy** below is real-image classification accuracy of its auxiliary class
head, not the probability that generated data looks real. The generator produces
six-dimensional MRI features, not full-resolution MRI images.

### Key assumptions
- The official `Testing` folder is held out. Validation comes only from `Training`.
- Corrupt files are excluded; identical decoded grayscale images with conflicting
  labels are quarantined. Exact duplicates are retained once, with test priority.
- Patient identifiers are not assumed from filenames. Image-level deduplication cannot
  establish a patient-independent split. Near duplicates need manual review.
- PCA/scalers are fitted on training images only. No test labels select checkpoints.
- This notebook uses a small CNN trained from scratch and compact quantum circuits.
  Stronger pretrained baselines, patient grouping, multiple seeds, and external data
  would be needed for a stronger research claim.

### Validation of this notebook
Validated on CPU with Python 3.12, PyTorch 2.8.0, and PennyLane 0.42.3: all 35 code
cells execute in order on four-class synthetic image fixtures; single- and two-seed
paths, batched angle/amplitude circuits, gradients, duplicate handling, and checkpoint
reloads were checked. Reloaded checkpoints reproduce the saved predictions.
**Full Kaggle MRI training was not run here because the dataset download was unavailable.**
Run it in your Colab session; no Kaggle accuracy is claimed in advance. The final cells
export the actual metrics, split manifests, checkpoints, and plots from your run.

## Setup
### Cell 1 — Install the tested quantum dependencies
Run in a fresh runtime before importing PennyLane. Existing Colab PyTorch, NumPy, SciPy,
pandas, scikit-learn, Pillow, and plotting packages are reused. No TensorFlow or
`torchvision` is required. If you imported a different PennyLane version earlier,
restart the runtime and start here.

In [ ]:
import sys, subprocess, importlib.metadata as metadata, importlib.util

required_versions = {
    "PennyLane": "0.42.3",
    "PennyLane-Lightning": "0.42.0",
    "autoray": "0.7.2",
    "kagglehub": "0.3.13",
}
install = []
for package, version in required_versions.items():
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        install.append(f"{package}=={version}")
for module, package in {
    "torch": "torch", "numpy": "numpy", "scipy": "scipy",
    "pandas": "pandas", "sklearn": "scikit-learn", "PIL": "Pillow",
    "matplotlib": "matplotlib", "seaborn": "seaborn", "tqdm": "tqdm",
}.items():
    if importlib.util.find_spec(module) is None:
        install.append(package)
if install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *install])
print("Dependency setup complete. Continue to Cell 2.")

### Cell 2 — Imports, deterministic seeds, and environment checks

In [ ]:
import os, json, random, hashlib, time, copy, re, platform, zipfile, shutil
from pathlib import Path
from collections import OrderedDict
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix, roc_auc_score,
    roc_curve, auc,
)
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
import pennylane as qml
import kagglehub

assert qml.__version__ == "0.42.3", (
    "A different PennyLane version is already loaded. Restart the runtime, then run Cell 1."
)
torch.set_default_dtype(torch.float32)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]
N_CLASSES = len(CLASS_NAMES)
CLASS_TO_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}
CLASSICAL_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
QUANTUM_DEVICE = torch.device("cpu")
try:
    COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    COLAB = False
sns.set_theme(style="whitegrid", context="notebook")
print("Python:", platform.python_version(), "| PyTorch:", torch.__version__)
print("PennyLane:", qml.__version__, "| Classical:", CLASSICAL_DEVICE,
      "| Quantum:", QUANTUM_DEVICE)
print("CUDA GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")

### Cell 3 — Experiment settings
`full`: all retained images, 20 epochs; `quick`: at most 160 training / 40 validation /
80 test images per class, 5 epochs; `smoke`: at most 16 / 4 / 8 per class, 2 epochs.
Early stopping uses validation **macro F1** with validation loss as a tie-breaker.
The split stays fixed across training seeds. Start with one seed; set
`SEEDS = [42, 123, 2026]` for repeated runs. Save to Drive if you need persistence
after Colab disconnects.

In [ ]:
RUN_MODE = "full"                 # "full", "quick", or "smoke"
DATA_SOURCE = "kagglehub"         # "kagglehub", "upload", or "folder"
DATASET_HANDLE = "masoudnickparvar/brain-tumor-mri-dataset/versions/1"
DATASET_FOLDER = ""               # for DATA_SOURCE="folder": parent of Training/Testing
ZIP_PATH = ""                     # optional existing ZIP path for DATA_SOURCE="upload"
USE_DRIVE = False                 # True mounts Google Drive and saves outputs there
SEEDS = [42]
SPLIT_SEED = 42
VALIDATION_FRACTION = 0.15
IMAGE_SIZE = 64
PCA_IMAGE_SIZE = 32
QUANV_IMAGE_SIZE = 16            # even: 16×16 → 8×8×4 quantum feature map
N_QUBITS = 6
QUANTUM_LAYERS = 3
QUANV_LAYERS = 2
QUANV_SEED = 2026
BATCH_SIZE = 64
QUANTUM_BATCH_SIZE = 32
PATCH_BATCH_SIZE = 1024
LEARNING_RATE = 0.001
VQC_LEARNING_RATE = 0.02
QGAN_LEARNING_RATE = 0.001
QGAN_CLASS_WEIGHT = 1.0
PATIENCE = 5

profiles = {
    "full":  {"epochs": 20, "gan_epochs": 20, "caps": (None, None, None)},
    "quick": {"epochs": 5,  "gan_epochs": 5,  "caps": (160, 40, 80)},
    "smoke": {"epochs": 2,  "gan_epochs": 2,  "caps": (16, 4, 8)},
}
assert RUN_MODE in profiles and DATA_SOURCE in {"kagglehub", "upload", "folder"}
assert len(SEEDS) > 0 and len(set(SEEDS)) == len(SEEDS)
assert N_QUBITS >= 2 and N_QUBITS <= 8, "Keep circuits small for Colab."
assert QUANV_IMAGE_SIZE >= 8 and QUANV_IMAGE_SIZE % 2 == 0
EPOCHS = profiles[RUN_MODE]["epochs"]
GAN_EPOCHS = profiles[RUN_MODE]["gan_epochs"]
SPLIT_CAPS = profiles[RUN_MODE]["caps"]
if USE_DRIVE:
    if not COLAB:
        raise RuntimeError("USE_DRIVE=True requires Google Colab.")
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/QML_MRI_Study")
else:
    OUTPUT_ROOT = (Path("/content") if COLAB else Path.cwd()) / "QML_MRI_Study"
RUN_DIR = OUTPUT_ROOT / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + RUN_MODE)
for subdirectory in ["plots", "checkpoints", "histories", "predictions", "manifests"]:
    (RUN_DIR / subdirectory).mkdir(parents=True, exist_ok=True)
seed_everything(SPLIT_SEED)
CONFIG = {
    "run_mode": RUN_MODE, "dataset": DATASET_HANDLE, "data_source": DATA_SOURCE,
    "seeds": SEEDS, "split_seed": SPLIT_SEED, "validation_fraction": VALIDATION_FRACTION,
    "image_size": IMAGE_SIZE, "pca_image_size": PCA_IMAGE_SIZE,
    "quanv_image_size": QUANV_IMAGE_SIZE, "n_qubits": N_QUBITS,
    "quantum_layers": QUANTUM_LAYERS, "quanv_layers": QUANV_LAYERS,
    "quanv_seed": QUANV_SEED, "batch_size": BATCH_SIZE,
    "quantum_batch_size": QUANTUM_BATCH_SIZE, "patch_batch_size": PATCH_BATCH_SIZE,
    "epochs": EPOCHS, "gan_epochs": GAN_EPOCHS, "patience": PATIENCE,
    "learning_rate": LEARNING_RATE, "vqc_learning_rate": VQC_LEARNING_RATE,
    "qgan_learning_rate": QGAN_LEARNING_RATE, "qgan_class_weight": QGAN_CLASS_WEIGHT,
    "quantum_device": "default.qubit", "quantum_shots": None,
    "class_names": CLASS_NAMES,
}
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
print("Mode:", RUN_MODE, "| Seeds:", SEEDS, "| Epochs:", EPOCHS)
print("Outputs:", RUN_DIR)

## Steps — Data acquisition and EDA
### Cell 4 — Download or load the dataset
The dataset version is pinned to version 1. Public Kaggle downloads normally do not
require a token. If network/authentication fails, use the ZIP upload option in Cell 3.
The loader supports the extra top-level folder sometimes found in downloaded ZIPs.

In [ ]:
def safely_extract_zip(zip_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError("Unsafe ZIP path: " + member.filename)
        archive.extractall(destination)
    return destination

def locate_dataset_root(path):
    path = Path(path).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(f"Dataset folder does not exist: {path}")
    candidates = [path]
    candidates.extend(p.parent for p in path.rglob("*")
                      if p.is_dir() and p.name.lower() == "training")
    roots = []
    for candidate in candidates:
        children = {p.name.lower(): p for p in candidate.iterdir() if p.is_dir()}
        if "training" in children and "testing" in children:
            if candidate not in roots:
                roots.append(candidate)
    if len(roots) != 1:
        raise ValueError(f"Expected one Training/Testing root, found {len(roots)}: {roots}. "
                         "Set DATASET_FOLDER to the exact dataset directory.")
    return roots[0]

if DATA_SOURCE == "kagglehub":
    try:
        download_path = kagglehub.dataset_download(DATASET_HANDLE)
    except Exception as exc:
        raise RuntimeError(
            "Kaggle download failed. Download the dataset ZIP in your browser; set "
            "DATA_SOURCE='upload' in Cell 3 and rerun Cells 3 onward. "
            f"Original error: {type(exc).__name__}: {exc}"
        ) from exc
elif DATA_SOURCE == "upload":
    if ZIP_PATH:
        selected_zip = Path(ZIP_PATH).expanduser()
    else:
        if not COLAB:
            raise RuntimeError("Outside Colab, set ZIP_PATH or use DATA_SOURCE='folder'.")
        from google.colab import files
        uploaded = files.upload()
        zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
        if len(zip_names) != 1:
            raise ValueError("Upload exactly one dataset ZIP.")
        selected_zip = Path(zip_names[0])
    download_path = safely_extract_zip(selected_zip, RUN_DIR / "downloaded_dataset")
else:
    if not DATASET_FOLDER:
        raise ValueError("Set DATASET_FOLDER in Cell 3.")
    download_path = DATASET_FOLDER
DATA_ROOT = locate_dataset_root(download_path)
split_directories = {p.name.lower(): p for p in DATA_ROOT.iterdir() if p.is_dir()}
print("Dataset root:", DATA_ROOT)
(RUN_DIR / "dataset_source.json").write_text(json.dumps({
    "handle": DATASET_HANDLE, "local_root": str(DATA_ROOT),
    "acquired_utc": datetime.now(timezone.utc).isoformat(),
}, indent=2))

### Cell 5 — Build and validate the image manifest
Inspect every file with Pillow. Hash decoded grayscale pixels plus dimensions, so
identical image content is detected even if filenames or file metadata differ.
A 64-bit difference hash flags possible near-duplicate groups for review; it is a
heuristic and is not automatically used to delete images.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
def canonical_class(name):
    normalized = re.sub(r"[^a-z]", "", name.lower())
    return {"notumor": "notumor", "no": "notumor"}.get(normalized, normalized)

records, bad_files = [], []
for split_key, official_split in [("training", "Training"), ("testing", "Testing")]:
    split_directory = split_directories[split_key]
    image_paths = sorted(p for p in split_directory.rglob("*")
                         if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    for image_path in tqdm(image_paths, desc=f"Inspect {official_split}"):
        label = canonical_class(image_path.relative_to(split_directory).parts[0])
        if label not in CLASS_TO_INDEX:
            raise ValueError(f"Unexpected class folder for {image_path}: {label}")
        try:
            with Image.open(image_path) as image:
                image.load()
                width, height, mode = image.width, image.height, image.mode
                gray_image = ImageOps.exif_transpose(image).convert("L")
                gray = np.asarray(gray_image, dtype=np.uint8)
                content_hash = hashlib.sha256(
                    np.asarray(gray.shape, dtype=np.int64).tobytes() + gray.tobytes()
                ).hexdigest()
                small = np.asarray(gray_image.resize((64, 64), Image.Resampling.BILINEAR))
                histogram = np.bincount(small.ravel(), minlength=256).astype(float)
                probability = histogram / histogram.sum()
                nonzero = probability > 0
                entropy = float(-(probability[nonzero] * np.log2(probability[nonzero])).sum())
                hash_image = np.asarray(gray_image.resize((9, 8), Image.Resampling.BILINEAR))
                difference_hash = np.packbits(hash_image[:, 1:] > hash_image[:, :-1]).tobytes().hex()
            records.append({
                "path": str(image_path), "relative_path": str(image_path.relative_to(DATA_ROOT)),
                "official_split": official_split, "label": label, "target": CLASS_TO_INDEX[label],
                "width": width, "height": height, "mode": mode,
                "file_bytes": image_path.stat().st_size, "sha256": content_hash,
                "dhash64": difference_hash, "mean_intensity": float(small.mean()),
                "std_intensity": float(small.std()), "entropy_bits": entropy,
                "low_contrast": bool(small.std() < 1.0),
            })
        except (OSError, ValueError, SyntaxError) as exc:
            bad_files.append({"path": str(image_path), "reason": f"{type(exc).__name__}: {exc}"})
raw_manifest = pd.DataFrame(records)
if raw_manifest.empty:
    raise ValueError("No readable labeled images found in Training/Testing.")
assert set(raw_manifest.label) == set(CLASS_NAMES), "All four classes must be present."
raw_manifest.to_csv(RUN_DIR / "manifests" / "raw_manifest.csv", index=False)
pd.DataFrame(bad_files, columns=["path", "reason"]).to_csv(
    RUN_DIR / "manifests" / "corrupt_files.csv", index=False)
print(f"Readable images: {len(raw_manifest):,}; corrupt files: {len(bad_files):,}")
display(raw_manifest.groupby(["official_split", "label"]).size().unstack(fill_value=0)
        .reindex(columns=CLASS_NAMES))
display(raw_manifest[["width", "height", "file_bytes", "mean_intensity",
                      "std_intensity", "entropy_bits"]].describe().round(2))
if bad_files:
    display(pd.DataFrame(bad_files).head(10))

### Cell 6 — Plot class balance and image quality

In [ ]:
def save_figure(figure, filename):
    figure.savefig(RUN_DIR / "plots" / filename, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(figure)

counts = raw_manifest.groupby(["label", "official_split"]).size().unstack(fill_value=0)
counts = counts.reindex(CLASS_NAMES)
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
counts.plot.bar(ax=axes[0, 0], color=["#6366f1", "#0f766e"], rot=15)
axes[0, 0].set(title="Raw class counts by official split", ylabel="Images", xlabel="Class")
axes[0, 1].scatter(raw_manifest.width, raw_manifest.height, alpha=0.2, s=12)
axes[0, 1].set(title="Original image dimensions", xlabel="Width (pixels)", ylabel="Height (pixels)")
sns.boxplot(data=raw_manifest, x="label", y="mean_intensity", order=CLASS_NAMES,
            ax=axes[1, 0], color="#c7d2fe")
axes[1, 0].set(title="Brightness by class", ylabel="Mean grayscale intensity", xlabel="Class")
sns.boxplot(data=raw_manifest, x="label", y="entropy_bits", order=CLASS_NAMES,
            ax=axes[1, 1], color="#a7f3d0")
axes[1, 1].set(title="Image entropy by class", ylabel="Entropy (bits)", xlabel="Class")
fig.tight_layout()
save_figure(fig, "01_eda_distribution_quality.png")
print("Low-contrast images flagged for manual review:", int(raw_manifest.low_contrast.sum()))

### Cell 7 — Display real dataset samples

In [ ]:
fig, axes = plt.subplots(N_CLASSES, 4, figsize=(10, 10), squeeze=False)
for row, label in enumerate(CLASS_NAMES):
    subset = raw_manifest[raw_manifest.label == label]
    samples = subset.sample(n=min(4, len(subset)), random_state=SPLIT_SEED)
    for column, record in enumerate(samples.itertuples()):
        with Image.open(record.path) as image:
            axes[row, column].imshow(ImageOps.exif_transpose(image).convert("L"), cmap="gray")
        axes[row, column].set_title(f"{label} | {record.official_split}", fontsize=9)
    for axis in axes[row]:
        axis.axis("off")
fig.suptitle("Random real images, before resizing", fontsize=15)
fig.tight_layout()
save_figure(fig, "02_raw_image_samples.png")

### Cell 8 — Audit duplicates and quarantine conflicting labels
This step runs before train/validation splitting. Preserve one test copy when the
same decoded image occurs in both official folders; remove its training copies.
The retained test set is therefore the **deduplicated official test set**, not
necessarily the original number of test files. Possible near duplicates are reported
but retained for manual review; a dHash match can be a false positive.

In [ ]:
hash_group = raw_manifest.groupby("sha256")
conflict_hashes = set(hash_group.label.nunique().loc[lambda series: series > 1].index)
exact_cross_split = hash_group.official_split.nunique().loc[lambda series: series > 1]
near_cross_split = raw_manifest.groupby("dhash64").official_split.nunique().loc[lambda series: series > 1]
conflicts = raw_manifest[raw_manifest.sha256.isin(conflict_hashes)].copy()
conflicts.to_csv(RUN_DIR / "manifests" / "conflicting_labels.csv", index=False)
duplicate_rows = raw_manifest[raw_manifest.duplicated("sha256", keep=False)].copy()
duplicate_rows.to_csv(RUN_DIR / "manifests" / "exact_duplicates.csv", index=False)
near_rows = raw_manifest[raw_manifest.dhash64.isin(near_cross_split.index)].copy()
near_rows.to_csv(RUN_DIR / "manifests" / "possible_near_duplicates.csv", index=False)

clean_manifest = raw_manifest[~raw_manifest.sha256.isin(conflict_hashes)].copy()
clean_manifest["priority"] = clean_manifest.official_split.map({"Testing": 0, "Training": 1})
clean_manifest = (clean_manifest.sort_values(["priority", "relative_path"])
                  .drop_duplicates("sha256", keep="first").drop(columns="priority")
                  .reset_index(drop=True))
assert not clean_manifest.sha256.duplicated().any()
print("Exact duplicate groups across official splits:", len(exact_cross_split))
print("Conflicting-label images quarantined:", len(conflicts))
print("Possible near-duplicate cross-split dHash groups:", len(near_cross_split))
print("Images retained after exact deduplication:", len(clean_manifest))
display(clean_manifest.groupby(["official_split", "label"]).size().unstack(fill_value=0)
        .reindex(columns=CLASS_NAMES))

if len(near_cross_split):
    review_hashes = list(near_cross_split.index[:3])
    fig, axes = plt.subplots(len(review_hashes), 2, figsize=(6, 3 * len(review_hashes)), squeeze=False)
    for row, image_hash in enumerate(review_hashes):
        candidates = near_rows[near_rows.dhash64 == image_hash]
        for column, split in enumerate(["Training", "Testing"]):
            example = candidates[candidates.official_split == split].iloc[0]
            with Image.open(example.path) as image:
                axes[row, column].imshow(image.convert("L"), cmap="gray")
            axes[row, column].set_title(f"{split}: {example.label}")
            axes[row, column].axis("off")
    fig.suptitle("Possible near duplicates: inspect manually")
    fig.tight_layout()
    save_figure(fig, "03_possible_near_duplicates.png")

### Cell 9 — Create a stratified validation split and optional subsets
All models use identical sample identities. The official test images never enter
training or PCA fitting. Exact content hashes are asserted disjoint in all three sets.
The split is at image level; without reliable subject IDs, patient independence remains
unverified. No augmentation is used so cached patches and image models share fixed inputs.

In [ ]:
train_pool = clean_manifest[clean_manifest.official_split == "Training"].copy()
test_manifest = clean_manifest[clean_manifest.official_split == "Testing"].copy()
for label in CLASS_NAMES:
    if (train_pool.label == label).sum() < 4 or (test_manifest.label == label).sum() < 1:
        raise ValueError(f"Not enough retained images for class {label}.")
validation_size = max(N_CLASSES, int(np.ceil(len(train_pool) * VALIDATION_FRACTION)))
validation_size = min(validation_size, len(train_pool) - N_CLASSES)
train_manifest, val_manifest = train_test_split(
    train_pool, test_size=validation_size, stratify=train_pool.target, random_state=SPLIT_SEED)

def cap_per_class(frame, cap):
    if cap is None:
        return frame.sort_values("relative_path").reset_index(drop=True)
    return pd.concat([
        group.sample(n=min(cap, len(group)), random_state=SPLIT_SEED)
        for _, group in frame.groupby("target", sort=True)
    ]).sort_values("relative_path").reset_index(drop=True)

train_manifest = cap_per_class(train_manifest, SPLIT_CAPS[0])
val_manifest = cap_per_class(val_manifest, SPLIT_CAPS[1])
test_manifest = cap_per_class(test_manifest, SPLIT_CAPS[2])
manifests = {"train": train_manifest, "val": val_manifest, "test": test_manifest}
for split, frame in manifests.items():
    assert set(frame.target) == set(range(N_CLASSES)), f"Class missing from {split}"
    frame.to_csv(RUN_DIR / "manifests" / f"{split}_manifest.csv", index=False)
for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
    assert set(manifests[left].sha256).isdisjoint(set(manifests[right].sha256))
split_counts = pd.DataFrame({
    split: frame.label.value_counts().reindex(CLASS_NAMES, fill_value=0)
    for split, frame in manifests.items()
})
display(split_counts)
split_counts.to_csv(RUN_DIR / "manifests" / "split_counts.csv")
print("PASS: four classes present, exact content hashes disjoint.")

### Cell 10 — Resize, preserve aspect ratio, and normalize
Grayscale images are letterboxed to 64×64 and scaled to [0,1]. Pixel values are not
standardized using test statistics. The smaller inputs below are deterministic resizes
of these same images. No tumor segmentation or localization is inferred from class labels.

In [ ]:
def load_image_tensor(frame, image_size):
    images = np.empty((len(frame), 1, image_size, image_size), dtype=np.float32)
    for index, path in enumerate(tqdm(frame.path, desc=f"Load {image_size}×{image_size}")):
        with Image.open(path) as image:
            gray = ImageOps.exif_transpose(image).convert("L")
            gray = ImageOps.pad(gray, (image_size, image_size),
                                method=Image.Resampling.BILINEAR, color=0, centering=(0.5, 0.5))
            images[index, 0] = np.asarray(gray, dtype=np.float32) / 255.0
    return torch.from_numpy(images)

image_tensors = {split: load_image_tensor(frame, IMAGE_SIZE) for split, frame in manifests.items()}
targets = {split: torch.tensor(frame.target.to_numpy(), dtype=torch.long)
           for split, frame in manifests.items()}
class_counts = np.bincount(targets["train"].numpy(), minlength=N_CLASSES)
CLASS_WEIGHTS = torch.tensor(len(targets["train"]) / (N_CLASSES * class_counts), dtype=torch.float32)
for split, images in image_tensors.items():
    assert images.ndim == 4 and images.shape[1] == 1
    assert torch.isfinite(images).all() and images.min() >= 0 and images.max() <= 1
    print(split, tuple(images.shape), "labels", tuple(targets[split].shape))
print("Training-only class weights:", dict(zip(CLASS_NAMES, CLASS_WEIGHTS.tolist())))

### Cell 11 — Fit PCA and feature scalers on training data only
Flatten 32×32 images, fit up to 64 principal components, and zero-pad only if a small
smoke run has fewer available components. The first six components are standardized
using training statistics and bounded with tanh. Angle inputs are π times these features.
Amplitude inputs are unit-length PCA vectors; a zero vector safely maps to |0…0⟩.
The amplitude MLP receives exactly the same normalized vectors as the amplitude QNN.

In [ ]:
flat_pixels = {
    split: F.interpolate(images, size=(PCA_IMAGE_SIZE, PCA_IMAGE_SIZE),
                         mode="bilinear", align_corners=False).flatten(1).numpy()
    for split, images in image_tensors.items()
}
AMPLITUDE_DIM = 2 ** N_QUBITS
pca_components = min(AMPLITUDE_DIM, len(flat_pixels["train"]) - 1, flat_pixels["train"].shape[1])
assert pca_components >= N_QUBITS, "Use more training images or fewer qubits."
pca = PCA(n_components=pca_components, svd_solver="randomized", random_state=SPLIT_SEED)
pca.fit(flat_pixels["train"])
pca_features = {split: pca.transform(pixels).astype(np.float32) for split, pixels in flat_pixels.items()}
angle_scaler = StandardScaler().fit(pca_features["train"][:, :N_QUBITS])
latent_features, angle_features, amplitude_features = {}, {}, {}
for split, features in pca_features.items():
    standardized = angle_scaler.transform(features[:, :N_QUBITS]).astype(np.float32)
    latent = np.tanh(standardized / 2.0).astype(np.float32)
    latent_features[split] = torch.from_numpy(latent)
    angle_features[split] = torch.from_numpy(np.pi * latent)
    padded = np.pad(features, ((0, 0), (0, AMPLITUDE_DIM - features.shape[1]))).astype(np.float32)
    norms = np.linalg.norm(padded, axis=1, keepdims=True)
    zero = norms[:, 0] < 1e-8
    padded[zero] = 0.0
    padded[zero, 0] = 1.0
    padded /= np.linalg.norm(padded, axis=1, keepdims=True)
    amplitude_features[split] = torch.from_numpy(padded)
    assert torch.isfinite(angle_features[split]).all()
    assert torch.allclose(amplitude_features[split].norm(dim=1),
                          torch.ones(len(features)), atol=1e-5)
np.savez(RUN_DIR / "pca_and_scaler.npz", pca_components=pca.components_, pca_mean=pca.mean_,
         explained_variance=pca.explained_variance_, explained_ratio=pca.explained_variance_ratio_,
         scaler_mean=angle_scaler.mean_, scaler_scale=angle_scaler.scale_)
fig, axis = plt.subplots(figsize=(8, 4))
axis.plot(np.arange(1, pca_components + 1), np.cumsum(pca.explained_variance_ratio_), marker=".")
axis.axvline(N_QUBITS, color="#dc2626", linestyle="--", label="Angle / QGAN feature count")
axis.set(xlabel="Number of PCA components", ylabel="Cumulative explained variance",
         title="PCA fitted on training images only", ylim=(0, 1.01))
axis.legend()
fig.tight_layout()
save_figure(fig, "04_pca_explained_variance.png")
print(f"Variance captured by {N_QUBITS} features: {pca.explained_variance_ratio_[:N_QUBITS].sum():.3f}")
print(f"Variance captured by {pca_components} features: {pca.explained_variance_ratio_.sum():.3f}")

## Steps — Quantum circuits and model definitions
### Cell 12 — Differentiable batched quantum layer
Every circuit uses real RY/RZ rotations and a CNOT ring, with exact measurements on
PennyLane `default.qubit`. Inputs and weights are copied to CPU float64 for numerical
stability; differentiable copies preserve gradients back to the CNN if it runs on GPU.
The layer converts measured outputs back to the caller's device and float32 dtype.
VQC re-encodes its angles at every layer; angle QNN encodes once; amplitude QNN
loads a normalized state once. Six qubits store 64 complex amplitudes.

In [ ]:
def make_quantum_node(n_qubits, n_layers, encoding="angle", reupload=False, readout="expectations"):
    device = qml.device("default.qubit", wires=n_qubits, shots=None)
    wires = list(range(n_qubits))
    @qml.qnode(device, interface="torch", diff_method="backprop")
    def circuit(inputs, weights):
        if encoding == "amplitude":
            qml.AmplitudeEmbedding(inputs, wires=wires, normalize=True)
        for layer in range(n_layers):
            if encoding == "angle" and (layer == 0 or reupload):
                for wire in wires:
                    qml.RY(inputs[..., wire], wires=wire)
            for wire in wires:
                qml.RY(weights[layer, wire, 0], wires=wire)
                qml.RZ(weights[layer, wire, 1], wires=wire)
            for wire in wires:
                qml.CNOT(wires=[wire, (wire + 1) % n_qubits])
        if readout == "probabilities":
            return qml.probs(wires=[0, 1])
        return [qml.expval(qml.PauliZ(wire)) for wire in wires]
    return circuit

class QuantumLayer(nn.Module):
    def __init__(self, n_qubits=N_QUBITS, n_layers=QUANTUM_LAYERS,
                 encoding="angle", reupload=False, readout="expectations"):
        super().__init__()
        if encoding not in {"angle", "amplitude"}:
            raise ValueError("Unknown quantum encoding.")
        self.encoding, self.readout = encoding, readout
        self.n_qubits, self.n_layers = n_qubits, n_layers
        self.weights = nn.Parameter(0.1 * torch.randn(n_layers, n_qubits, 2))
        self.qnode = make_quantum_node(n_qubits, n_layers, encoding, reupload, readout)

    def forward(self, inputs):
        if inputs.ndim == 1:
            inputs = inputs.unsqueeze(0)
        expected = 2 ** self.n_qubits if self.encoding == "amplitude" else self.n_qubits
        if inputs.ndim != 2 or inputs.shape[-1] != expected:
            raise ValueError(f"Expected a batch with {expected} features; got {tuple(inputs.shape)}")
        quantum_inputs = inputs.to(device=QUANTUM_DEVICE, dtype=torch.float64)
        quantum_weights = self.weights.to(device=QUANTUM_DEVICE, dtype=torch.float64)
        result = self.qnode(quantum_inputs, quantum_weights)
        if self.readout == "expectations":
            result = torch.stack(result, dim=-1)
        return result.to(device=inputs.device, dtype=inputs.dtype)

class VariationalQuantumClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.quantum = QuantumLayer(reupload=True, readout="probabilities")
    def forward(self, angles):
        probabilities = self.quantum(angles)
        # CrossEntropyLoss accepts these normalized log-probabilities as logits.
        return probabilities.clamp_min(1e-8).log()

class QuantumNeuralNetwork(nn.Module):
    def __init__(self, encoding):
        super().__init__()
        self.quantum = QuantumLayer(encoding=encoding, reupload=False)
        self.classifier = nn.Linear(N_QUBITS, N_CLASSES)
    def forward(self, features):
        return self.classifier(self.quantum(features))

print("Quantum layer defined: batch dimensions and devices are explicit.")

### Cell 13 — Classical controls and jointly trained hybrid CNN
The hybrid circuit replaces a classical six-dimensional transform. The CNN bottleneck
control uses the same feature extractor and projection, plus a classical nonlinear layer.
The latent MLP shares the QGAN discriminator's trunk and class head; the adversarial
head is the QGAN-specific addition. No ImageNet weights or downloads are required.

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64), nn.ReLU(),
        )
    def forward(self, images):
        return self.layers(images)

class ClassicalCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ImageEncoder()
        self.classifier = nn.Linear(64, N_CLASSES)
    def forward(self, images):
        return self.classifier(self.encoder(images))

class BottleneckCNN(nn.Module):
    def __init__(self, use_quantum=False):
        super().__init__()
        self.encoder = ImageEncoder()
        self.projection = nn.Linear(64, N_QUBITS)
        self.transform = (QuantumLayer() if use_quantum else
                          nn.Sequential(nn.Linear(N_QUBITS, N_QUBITS), nn.Tanh()))
        self.classifier = nn.Linear(N_QUBITS, N_CLASSES)
        self.use_quantum = use_quantum
    def forward(self, images):
        bounded = torch.tanh(self.projection(self.encoder(images)))
        features = self.transform(np.pi * bounded if self.use_quantum else bounded)
        return self.classifier(features)

class LatentMLP(nn.Module):
    def __init__(self, input_dim=N_QUBITS):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(),
                                   nn.Linear(64, 32), nn.ReLU())
        self.classifier = nn.Linear(32, N_CLASSES)
    def forward(self, features):
        return self.classifier(self.trunk(features))

class PatchClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(4, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(),
            nn.Linear(32 * 4 * 4, 64), nn.ReLU(), nn.Linear(64, N_CLASSES),
        )
    def forward(self, patches):
        return self.layers(patches)

class ConditionalQuantumGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.label_embedding = nn.Embedding(N_CLASSES, N_QUBITS)
        nn.init.normal_(self.label_embedding.weight, std=0.2)
        self.quantum = QuantumLayer(reupload=True)
    def forward(self, noise, labels):
        angles = np.pi * torch.tanh(noise + self.label_embedding(labels))
        # Z expectations already lie in [-1,1], matching the real latent features.
        return self.quantum(angles)

class AuxiliaryDiscriminator(LatentMLP):
    def __init__(self):
        super().__init__(N_QUBITS)
        self.adversarial = nn.Linear(32, 1)
    def discriminate(self, features):
        hidden = self.trunk(features)
        return self.adversarial(hidden).squeeze(-1), self.classifier(hidden)

print("Defined classical CNN, bottleneck CNN, hybrid CNN, MLPs, patch CNN, and conditional QGAN.")

### Cell 14 — Precompute quanvolution and its patch control
Each 16×16 image yields 64 non-overlapping 2×2 patches. The same fixed four-qubit
circuit transforms all patches; its four Z expectation values form an 8×8×4 feature map.
The classical control receives the same four raw pixels, rearranged into an 8×8×4
map and rescaled to [-1,1]. Both use the identical `PatchClassifier` head.
Cache computation is batched and runs once per dataset split, not every epoch.

In [ ]:
kernel_rng = torch.Generator().manual_seed(QUANV_SEED)
quanv_weights = (2 * np.pi * torch.rand(QUANV_LAYERS, 4, 2, generator=kernel_rng)
                  - np.pi).double()
quanv_node = make_quantum_node(4, QUANV_LAYERS, "angle", False, "expectations")
quanv_features, patch_features = {}, {}
quanv_preprocessing_seconds = {}
grid_size = QUANV_IMAGE_SIZE // 2
for split, images in image_tensors.items():
    start_time = time.perf_counter()
    tiny = F.interpolate(images, size=(QUANV_IMAGE_SIZE, QUANV_IMAGE_SIZE),
                         mode="bilinear", align_corners=False)
    patches = F.unfold(tiny, kernel_size=2, stride=2)   # N×4×number_of_patches
    patch_features[split] = (2 * patches - 1).reshape(len(images), 4, grid_size, grid_size).contiguous()
    flat_patches = patches.transpose(1, 2).reshape(-1, 4)
    measured_batches = []
    with torch.no_grad():
        for begin in tqdm(range(0, len(flat_patches), PATCH_BATCH_SIZE), desc=f"Quanvolve {split}"):
            angles = (np.pi * flat_patches[begin:begin + PATCH_BATCH_SIZE]).double()
            measured = torch.stack(quanv_node(angles, quanv_weights), dim=-1).float()
            measured_batches.append(measured)
    measured = torch.cat(measured_batches)
    quanv_features[split] = (measured.reshape(len(images), grid_size * grid_size, 4)
                             .transpose(1, 2).reshape(len(images), 4, grid_size, grid_size).contiguous())
    assert torch.isfinite(quanv_features[split]).all()
    assert quanv_features[split].abs().max() <= 1.00001
    quanv_preprocessing_seconds[split] = time.perf_counter() - start_time
    torch.save(quanv_features[split], RUN_DIR / f"quanvolution_{split}.pt")
    print(split, tuple(quanv_features[split].shape),
          f"| preprocessing {quanv_preprocessing_seconds[split]:.1f} s")
torch.save(quanv_weights, RUN_DIR / "quanvolution_kernel_weights.pt")
fig, axes = plt.subplots(1, 5, figsize=(13, 3))
axes[0].imshow(image_tensors["train"][0, 0], cmap="gray")
axes[0].set_title("Input MRI")
for channel in range(4):
    axes[channel + 1].imshow(quanv_features["train"][0, channel], cmap="viridis", vmin=-1, vmax=1)
    axes[channel + 1].set_title(f"Quantum channel {channel + 1}")
for axis in axes:
    axis.axis("off")
fig.tight_layout()
save_figure(fig, "05_quanvolution_channels.png")

## Checks — Verify before long training
### Cell 15 — Check circuit values, gradients, shapes, and batch size one
A one-qubit rotation must satisfy ⟨Z⟩ = cos(angle); a Bell state must have four-state
probabilities [0.5,0,0,0.5]. Every trainable quantum model must yield finite outputs
and nonzero circuit gradients. The hybrid must also backpropagate into its CNN.

In [ ]:
analytic_device = qml.device("default.qubit", wires=2)
@qml.qnode(analytic_device, interface="torch", diff_method="backprop")
def rotation_check(angle):
    qml.RY(angle, wires=0)
    return qml.expval(qml.PauliZ(0))
probe_angle = torch.tensor(0.41, dtype=torch.float64, requires_grad=True)
probe_result = rotation_check(probe_angle)
assert torch.allclose(probe_result, probe_angle.cos(), atol=1e-10)
probe_result.backward()
assert torch.allclose(probe_angle.grad, -probe_angle.detach().sin(), atol=1e-10)
@qml.qnode(analytic_device, interface="torch")
def bell_check():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.probs(wires=[0, 1])
assert np.allclose(np.asarray(bell_check()), [0.5, 0, 0, 0.5], atol=1e-10)

seed_everything(SPLIT_SEED)
checks = [
    ("VQC", VariationalQuantumClassifier(), angle_features["train"][:3], QUANTUM_DEVICE),
    ("Angle QNN", QuantumNeuralNetwork("angle"), angle_features["train"][:3], QUANTUM_DEVICE),
    ("Amplitude QNN", QuantumNeuralNetwork("amplitude"), amplitude_features["train"][:3], QUANTUM_DEVICE),
    ("Hybrid CNN", BottleneckCNN(True), image_tensors["train"][:3], CLASSICAL_DEVICE),
]
for name, model, batch, device in checks:
    model = model.to(device)
    batch = batch.to(device)
    scores = model(batch)
    assert scores.shape == (len(batch), N_CLASSES) and torch.isfinite(scores).all(), name
    F.cross_entropy(scores, targets["train"][:len(batch)].to(device)).backward()
    qgrad = model.quantum.weights.grad if hasattr(model, "quantum") else model.transform.weights.grad
    assert qgrad is not None and torch.isfinite(qgrad).all() and qgrad.abs().sum() > 1e-12, name
    if name == "Hybrid CNN":
        assert model.encoder.layers[0].weight.grad.abs().sum() > 1e-12
    with torch.no_grad():
        assert model(batch[:1]).shape == (1, N_CLASSES), f"Batch-size-one error: {name}"
    print("PASS:", name, "| logits", tuple(scores.shape), "| finite, nonzero quantum gradients")

generator = ConditionalQuantumGenerator()
discriminator = AuxiliaryDiscriminator()
noise = torch.randn(3, N_QUBITS)
fake = generator(noise, targets["train"][:3])
fake_score, fake_class = discriminator.discriminate(fake)
(F.binary_cross_entropy_with_logits(fake_score, torch.ones(3))
 + F.cross_entropy(fake_class, targets["train"][:3])).backward()
assert generator.quantum.weights.grad.abs().sum() > 1e-12
assert generator.label_embedding.weight.grad.abs().sum() > 1e-12
assert fake.shape == (3, N_QUBITS) and torch.isfinite(fake).all()
print("PASS: QGAN generator and conditioning gradients.")
print("PASS: analytic rotation, analytic derivative, Bell state, normalized amplitude inputs.")
del checks, model, generator, discriminator, batch, scores, fake, fake_score, fake_class
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## Steps — Training and evaluation utilities
### Cell 16 — Data loaders, metrics, and classifier training
Class-weighted cross entropy uses training labels only. Validation loss is unweighted
cross entropy. Evaluate the test set only after restoring the best validation checkpoint.
Training curves use actual epoch outputs. Early stopping may make curve lengths differ.

In [ ]:
FEATURE_SETS = {
    "images": image_tensors, "latent": latent_features, "angles": angle_features,
    "amplitudes": amplitude_features, "quanvolution": quanv_features, "patches": patch_features,
}

def safe_name(name):
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")

def make_loader(features, labels, batch_size, shuffle=False, seed=SPLIT_SEED):
    return DataLoader(TensorDataset(features, labels), batch_size=batch_size,
                      shuffle=shuffle, num_workers=0, drop_last=False,
                      generator=torch.Generator().manual_seed(seed))

def model_batch_size(name, feature_key):
    # Quantum models and their corresponding controls use the same batch size.
    if feature_key in {"latent", "angles", "amplitudes"} or name in {
        "Hybrid quantum CNN", "CNN bottleneck control"
    }:
        return QUANTUM_BATCH_SIZE
    return BATCH_SIZE

def weighted_cross_entropy(logits, labels, weights):
    numerator = F.cross_entropy(logits, labels, weight=weights, reduction="none").sum()
    denominator = weights[labels].sum().clamp_min(1e-8)
    return numerator / denominator

def evaluate_classifier(model, loader):
    device = next(model.parameters()).device
    model.eval()
    probabilities, true_labels = [], []
    total_loss, total_count = 0.0, 0
    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device), labels.to(device)
            logits = model(features)
            if logits.shape != (len(labels), N_CLASSES) or not torch.isfinite(logits).all():
                raise RuntimeError("Classifier returned invalid logits.")
            total_loss += F.cross_entropy(logits, labels, reduction="sum").item()
            total_count += len(labels)
            probabilities.append(torch.softmax(logits, dim=1).cpu().numpy())
            true_labels.append(labels.cpu().numpy())
    probability = np.concatenate(probabilities)
    truth = np.concatenate(true_labels)
    predicted = probability.argmax(axis=1)
    scores = {
        "loss": total_loss / total_count,
        "accuracy": accuracy_score(truth, predicted),
        "balanced_accuracy": balanced_accuracy_score(truth, predicted),
        "macro_f1": f1_score(truth, predicted, labels=range(N_CLASSES), average="macro", zero_division=0),
        "macro_precision": precision_score(truth, predicted, labels=range(N_CLASSES), average="macro", zero_division=0),
        "macro_recall": recall_score(truth, predicted, labels=range(N_CLASSES), average="macro", zero_division=0),
    }
    try:
        scores["macro_auc"] = roc_auc_score(truth, probability, labels=range(N_CLASSES),
                                           multi_class="ovr", average="macro")
    except ValueError:
        scores["macro_auc"] = np.nan
    return scores, truth, predicted, probability

def train_classifier(name, factory, feature_key, seed, device, batch_size, learning_rate):
    seed_everything(seed)
    model = factory().to(device)
    loaders = {split: make_loader(FEATURE_SETS[feature_key][split], targets[split], batch_size,
                                  shuffle=(split == "train"), seed=seed) for split in manifests}
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
    weights = CLASS_WEIGHTS.to(device)
    history, best_state = [], None
    best_f1, best_loss, best_epoch, patience_used = -np.inf, np.inf, 0, 0
    start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_numerator, loss_denominator, correct, total = 0.0, 0.0, 0, 0
        for features, labels in loaders["train"]:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(features)
            loss = weighted_cross_entropy(logits, labels, weights)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss in {name}; lower its learning rate.")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            denominator = weights[labels].sum().item()
            loss_numerator += loss.item() * denominator
            loss_denominator += denominator
            correct += (logits.argmax(1) == labels).sum().item()
            total += len(labels)
        validation, *_ = evaluate_classifier(model, loaders["val"])
        row = {"epoch": epoch, "train_loss": loss_numerator / loss_denominator,
               "train_accuracy": correct / total, "val_loss": validation["loss"],
               "val_accuracy": validation["accuracy"], "val_macro_f1": validation["macro_f1"]}
        history.append(row)
        print(f"{name} | seed {seed} | {epoch:02d}/{EPOCHS} | "
              f"train acc {row['train_accuracy']:.3f} | val acc {row['val_accuracy']:.3f} | "
              f"val macro F1 {row['val_macro_f1']:.3f}")
        improved = (validation["macro_f1"] > best_f1 + 1e-8 or
                    (abs(validation["macro_f1"] - best_f1) <= 1e-8 and validation["loss"] < best_loss))
        if improved:
            best_f1, best_loss, best_epoch = validation["macro_f1"], validation["loss"], epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_used = 0
        else:
            patience_used += 1
        if patience_used >= PATIENCE:
            print("Early stopping; restoring validation-selected epoch", best_epoch)
            break
    training_seconds = time.perf_counter() - start
    assert best_state is not None
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch, training_seconds

RESULTS, HISTORIES, PREDICTIONS, TRAINED_MODELS, GAN_GENERATORS = [], {}, {}, {}, {}
print("Training utilities ready.")

### Cell 17 — Record validation-selected models and held-out predictions
Accuracy confidence intervals use Wilson's binomial interval for each individual run.
They describe image-level uncertainty, not patient-level or cross-seed uncertainty.
Quantum preprocessing time is recorded separately from model training time.

In [ ]:
def wilson_interval(correct, n, z=1.96):
    p = correct / n
    denominator = 1 + z * z / n
    center = (p + z * z / (2 * n)) / denominator
    radius = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denominator
    return max(0.0, center - radius), min(1.0, center + radius)

def record_result(name, family, feature_key, seed, model, history, best_epoch, seconds,
                  quantum_parameter_count=None):
    key = (name, seed)
    if key in TRAINED_MODELS:
        raise RuntimeError(f"Already recorded {key}. Restart from Cell 16 before repeating training cells.")
    batch_size = model_batch_size(name, feature_key)
    validation_loader = make_loader(FEATURE_SETS[feature_key]["val"], targets["val"], batch_size)
    test_loader = make_loader(FEATURE_SETS[feature_key]["test"], targets["test"], batch_size)
    validation_scores, *_ = evaluate_classifier(model, validation_loader)
    test_scores, truth, predicted, probability = evaluate_classifier(model, test_loader)
    low, high = wilson_interval(int((truth == predicted).sum()), len(truth))
    parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    if quantum_parameter_count is None:
        quantum_parameter_count = sum(module.weights.numel() for module in model.modules()
                                      if isinstance(module, QuantumLayer))
    RESULTS.append({
        "model": name, "family": family, "seed": seed, "features": feature_key,
        "n_train": len(targets["train"]), "n_val": len(targets["val"]), "n_test": len(truth),
        "best_epoch": best_epoch, "batch_size": batch_size, "val_accuracy": validation_scores["accuracy"],
        "val_macro_f1": validation_scores["macro_f1"],
        "test_accuracy": test_scores["accuracy"], "accuracy_ci_low": low, "accuracy_ci_high": high,
        "test_balanced_accuracy": test_scores["balanced_accuracy"], "test_macro_f1": test_scores["macro_f1"],
        "test_macro_precision": test_scores["macro_precision"], "test_macro_recall": test_scores["macro_recall"],
        "test_macro_auc": test_scores["macro_auc"], "test_loss": test_scores["loss"],
        "classifier_trainable_parameters": parameter_count,
        "quantum_trainable_parameters": quantum_parameter_count, "training_seconds": seconds,
        "quantum_preprocessing_seconds": (sum(quanv_preprocessing_seconds.values())
                                          if feature_key == "quanvolution" else 0.0),
    })
    filename = f"{safe_name(name)}_seed_{seed}"
    history.to_csv(RUN_DIR / "histories" / f"{filename}.csv", index=False)
    report = classification_report(truth, predicted, labels=range(N_CLASSES),
                                   target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    (RUN_DIR / "predictions" / f"{filename}_report.json").write_text(json.dumps(report, indent=2))
    prediction_frame = test_manifest[["relative_path", "sha256", "label", "target"]].copy()
    prediction_frame["predicted_target"] = predicted
    prediction_frame["predicted_label"] = [CLASS_NAMES[value] for value in predicted]
    for index, label in enumerate(CLASS_NAMES):
        prediction_frame[f"probability_{label}"] = probability[:, index]
    prediction_frame.to_csv(RUN_DIR / "predictions" / f"{filename}_predictions.csv", index=False)
    torch.save({"model_name": name, "seed": seed, "class_names": CLASS_NAMES,
                "best_epoch": best_epoch, "feature_key": feature_key,
                "state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
                "config": CONFIG}, RUN_DIR / "checkpoints" / f"{filename}.pt")
    HISTORIES[key] = history
    PREDICTIONS[key] = {"truth": truth, "predicted": predicted, "probability": probability}
    TRAINED_MODELS[key] = model.cpu()
    pd.DataFrame(RESULTS).to_csv(RUN_DIR / "metrics_by_run.csv", index=False)
    print(f"Held-out result: {name} | accuracy {test_scores['accuracy']:.2%} | "
          f"macro F1 {test_scores['macro_f1']:.3f} | best epoch {best_epoch}")

def run_classifier(name, family, factory, feature_key, device, learning_rate=LEARNING_RATE):
    batch_size = model_batch_size(name, feature_key)
    for seed in SEEDS:
        model, history, best_epoch, seconds = train_classifier(
            name, factory, feature_key, seed, device, batch_size, learning_rate)
        record_result(name, family, feature_key, seed, model, history, best_epoch, seconds)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

### Cell 18 — Conditional latent-space QGAN training
The quantum generator receives noise and a learned class embedding. Its outputs are
six Z expectations. The classical discriminator has real/fake and four-class heads.
It is trained with adversarial loss plus supervised real-class loss and a smaller
fake-class term. The generator tries to fool the real/fake head and match its condition.
The best discriminator **validation macro F1** selects both states from the same epoch.
The generative loss curves are kept separate from classifier cross-entropy curves.

In [ ]:
def train_qgan(seed):
    seed_everything(seed)
    generator = ConditionalQuantumGenerator().to(QUANTUM_DEVICE)
    # Match the initialization of the latent MLP trunk and class head.
    seed_everything(seed)
    discriminator = AuxiliaryDiscriminator().to(QUANTUM_DEVICE)
    train_loader = make_loader(latent_features["train"], targets["train"],
                               QUANTUM_BATCH_SIZE, shuffle=True, seed=seed)
    val_loader = make_loader(latent_features["val"], targets["val"], QUANTUM_BATCH_SIZE)
    optimizer_g = torch.optim.Adam(generator.parameters(), lr=QGAN_LEARNING_RATE, betas=(0.5, 0.999))
    optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=QGAN_LEARNING_RATE, betas=(0.5, 0.999))
    weights = CLASS_WEIGHTS.to(QUANTUM_DEVICE)
    history, best_d, best_g = [], None, None
    best_f1, best_loss, best_epoch, patience_used = -np.inf, np.inf, 0, 0
    start = time.perf_counter()
    for epoch in range(1, GAN_EPOCHS + 1):
        generator.train()
        discriminator.train()
        d_total, g_total, class_numerator, class_denominator, correct, total = 0., 0., 0., 0., 0, 0
        for real, labels in train_loader:
            real, labels = real.to(QUANTUM_DEVICE), labels.to(QUANTUM_DEVICE)
            n = len(labels)
            fake_labels = torch.randint(N_CLASSES, (n,), device=QUANTUM_DEVICE)
            with torch.no_grad():
                fake = generator(torch.randn(n, N_QUBITS, device=QUANTUM_DEVICE), fake_labels)
            optimizer_d.zero_grad(set_to_none=True)
            real_score, real_class = discriminator.discriminate(real)
            fake_score, fake_class = discriminator.discriminate(fake)
            class_loss = weighted_cross_entropy(real_class, labels, weights)
            adversarial_loss = (
                F.binary_cross_entropy_with_logits(real_score, torch.full_like(real_score, 0.9))
                + F.binary_cross_entropy_with_logits(fake_score, torch.zeros_like(fake_score))
            )
            d_loss = adversarial_loss + QGAN_CLASS_WEIGHT * (
                class_loss + 0.25 * F.cross_entropy(fake_class, fake_labels))
            if not torch.isfinite(d_loss):
                raise RuntimeError("Non-finite QGAN discriminator loss; lower QGAN_LEARNING_RATE.")
            d_loss.backward()
            nn.utils.clip_grad_norm_(discriminator.parameters(), 5.0)
            optimizer_d.step()

            for parameter in discriminator.parameters():
                parameter.requires_grad_(False)
            optimizer_g.zero_grad(set_to_none=True)
            generated = generator(torch.randn(n, N_QUBITS, device=QUANTUM_DEVICE), fake_labels)
            generated_score, generated_class = discriminator.discriminate(generated)
            g_loss = (F.binary_cross_entropy_with_logits(generated_score, torch.ones_like(generated_score))
                      + QGAN_CLASS_WEIGHT * F.cross_entropy(generated_class, fake_labels))
            if not torch.isfinite(g_loss):
                raise RuntimeError("Non-finite QGAN generator loss; lower QGAN_LEARNING_RATE.")
            g_loss.backward()
            nn.utils.clip_grad_norm_(generator.parameters(), 5.0)
            optimizer_g.step()
            for parameter in discriminator.parameters():
                parameter.requires_grad_(True)

            denominator = weights[labels].sum().item()
            class_numerator += class_loss.item() * denominator
            class_denominator += denominator
            d_total += d_loss.item() * n
            g_total += g_loss.item() * n
            correct += (real_class.argmax(1) == labels).sum().item()
            total += n
        validation, *_ = evaluate_classifier(discriminator, val_loader)
        history.append({"epoch": epoch, "train_loss": class_numerator / class_denominator,
                        "train_accuracy": correct / total, "val_loss": validation["loss"],
                        "val_accuracy": validation["accuracy"], "val_macro_f1": validation["macro_f1"],
                        "discriminator_loss": d_total / total, "generator_loss": g_total / total})
        print(f"QGAN | seed {seed} | {epoch:02d}/{GAN_EPOCHS} | D {d_total / total:.3f} | "
              f"G {g_total / total:.3f} | val acc {validation['accuracy']:.3f} | "
              f"val macro F1 {validation['macro_f1']:.3f}")
        improved = (validation["macro_f1"] > best_f1 + 1e-8 or
                    (abs(validation["macro_f1"] - best_f1) <= 1e-8 and validation["loss"] < best_loss))
        if improved:
            best_f1, best_loss, best_epoch = validation["macro_f1"], validation["loss"], epoch
            best_d = {key: value.detach().cpu().clone() for key, value in discriminator.state_dict().items()}
            best_g = {key: value.detach().cpu().clone() for key, value in generator.state_dict().items()}
            patience_used = 0
        else:
            patience_used += 1
        if patience_used >= PATIENCE:
            break
    seconds = time.perf_counter() - start
    assert best_d is not None and best_g is not None
    discriminator.load_state_dict(best_d)
    generator.load_state_dict(best_g)
    return discriminator, generator, pd.DataFrame(history), best_epoch, seconds

### Cell 19 — Train all classical controls
The latent MLP uses the same bounded PCA inputs and class-head architecture as QGAN.
VQC and angle QNN use π times these features because their encoding uses rotation angles.
The amplitude MLP uses the exact normalized amplitude vector seen by amplitude QNN.
The bottleneck CNN isolates replacement of a classical transform by a quantum circuit.

In [ ]:
run_classifier("Classical CNN", "classical", ClassicalCNN, "images", CLASSICAL_DEVICE)
run_classifier("CNN bottleneck control", "classical", lambda: BottleneckCNN(False),
               "images", CLASSICAL_DEVICE)
run_classifier("Latent MLP control", "classical", lambda: LatentMLP(N_QUBITS),
               "latent", QUANTUM_DEVICE)
run_classifier("Amplitude MLP control", "classical", lambda: LatentMLP(AMPLITUDE_DIM),
               "amplitudes", QUANTUM_DEVICE)
run_classifier("Patch CNN control", "classical", PatchClassifier, "patches", CLASSICAL_DEVICE)

### Cell 20 — Train the quanvolution CNN head

In [ ]:
run_classifier("Quanvolution", "quantum", PatchClassifier,
               "quanvolution", CLASSICAL_DEVICE)

### Cell 21 — Train the hybrid quantum-classical CNN end to end

In [ ]:
run_classifier("Hybrid quantum CNN", "quantum", lambda: BottleneckCNN(True),
               "images", CLASSICAL_DEVICE)

### Cell 22 — Train the variational quantum classifier

In [ ]:
run_classifier("VQC", "quantum", VariationalQuantumClassifier,
               "angles", QUANTUM_DEVICE, learning_rate=VQC_LEARNING_RATE)

### Cell 23 — Train QGAN and evaluate its tumor-classification head

In [ ]:
for seed in SEEDS:
    discriminator, generator, history, best_epoch, seconds = train_qgan(seed)
    quantum_count = generator.quantum.weights.numel()
    record_result("QGAN + auxiliary classifier", "quantum", "latent", seed,
                  discriminator, history, best_epoch, seconds, quantum_count)
    RESULTS[-1]["generator_trainable_parameters"] = sum(p.numel() for p in generator.parameters())
    GAN_GENERATORS[seed] = generator.cpu()
    torch.save({"seed": seed, "best_epoch": best_epoch, "config": CONFIG,
                "state_dict": {key: value.detach().cpu() for key, value in generator.state_dict().items()}},
               RUN_DIR / "checkpoints" / f"qgan_generator_seed_{seed}.pt")

### Cell 24 — Train the angle-encoded quantum neural network

In [ ]:
run_classifier("Angle QNN", "quantum", lambda: QuantumNeuralNetwork("angle"),
               "angles", QUANTUM_DEVICE)

### Cell 25 — Train the amplitude-encoded quantum neural network

In [ ]:
run_classifier("Amplitude QNN", "quantum", lambda: QuantumNeuralNetwork("amplitude"),
               "amplitudes", QUANTUM_DEVICE)

## Results
### Cell 26 — Tabulate observed validation and test results
For one seed, seed standard deviations are shown as zero, not as evidence of stability.
For multiple seeds, report the mean and standard deviation over all seeds; do not keep
only the best test seed. Timing is simulator time and is environment-dependent.
QGAN parameter counts distinguish its discriminator classifier from its generator.

In [ ]:
results = pd.DataFrame(RESULTS)
expected_names = ["Classical CNN", "CNN bottleneck control", "Latent MLP control",
                  "Amplitude MLP control", "Patch CNN control", "Quanvolution",
                  "Hybrid quantum CNN", "VQC", "QGAN + auxiliary classifier", "Angle QNN", "Amplitude QNN"]
assert len(results) == len(expected_names) * len(SEEDS), "Some training cells have not finished."
assert set(results.model) == set(expected_names)
assert not results.duplicated(["model", "seed"]).any()
for column in ["test_accuracy", "test_macro_f1", "val_accuracy", "val_macro_f1"]:
    assert results[column].between(0, 1).all() and results[column].notna().all()
results["generator_trainable_parameters"] = results.get("generator_trainable_parameters", 0)
results["generator_trainable_parameters"] = results["generator_trainable_parameters"].fillna(0).astype(int)
results.to_csv(RUN_DIR / "metrics_by_run.csv", index=False)
summary = results.groupby("model", sort=False).agg(
    family=("family", "first"), runs=("seed", "count"),
    validation_accuracy_mean=("val_accuracy", "mean"),
    validation_macro_f1_mean=("val_macro_f1", "mean"),
    accuracy_mean=("test_accuracy", "mean"), accuracy_std=("test_accuracy", "std"),
    macro_f1_mean=("test_macro_f1", "mean"), macro_f1_std=("test_macro_f1", "std"),
    balanced_accuracy_mean=("test_balanced_accuracy", "mean"),
    macro_auc_mean=("test_macro_auc", "mean"), training_seconds_mean=("training_seconds", "mean"),
    quantum_preprocessing_seconds=("quantum_preprocessing_seconds", "first"),
    classifier_parameters=("classifier_trainable_parameters", "first"),
    quantum_parameters=("quantum_trainable_parameters", "first"),
    generator_parameters=("generator_trainable_parameters", "first"),
).reset_index()
summary[["accuracy_std", "macro_f1_std"]] = summary[["accuracy_std", "macro_f1_std"]].fillna(0)
summary.to_csv(RUN_DIR / "metrics_summary.csv", index=False)
display_columns = ["model", "runs", "validation_accuracy_mean", "accuracy_mean", "accuracy_std",
                   "macro_f1_mean", "macro_f1_std", "balanced_accuracy_mean", "macro_auc_mean"]
display(summary[display_columns].round(4))
display(summary[["model", "classifier_parameters", "quantum_parameters", "generator_parameters",
                 "training_seconds_mean", "quantum_preprocessing_seconds"]].round(2))
print("QGAN quantum parameters belong to its generator. Quanvolution has zero trainable circuit parameters.")
if RUN_MODE != "full":
    print("SUBSET RUN: these results are for a limited exploratory/smoke subset, not the full dataset.")

### Cell 27 — Plot test accuracy and macro F1
With one seed, accuracy error bars are 95% Wilson intervals over test images.
With several seeds, error bars are ± one seed standard deviation.
Macro F1 shows seed standard deviation only; it has no Wilson interval.
Models are shown in a fixed order, not ranked using the test set.

In [ ]:
plot_order = ["Quanvolution", "Hybrid quantum CNN", "VQC", "QGAN + auxiliary classifier",
              "Angle QNN", "Amplitude QNN", "Classical CNN", "CNN bottleneck control",
              "Latent MLP control", "Amplitude MLP control", "Patch CNN control"]
plot_summary = summary.set_index("model").loc[plot_order]
plot_labels = [name.replace(" + auxiliary classifier", " + class head") for name in plot_order]
colors = ["#0f766e" if family == "quantum" else "#6366f1" for family in plot_summary.family]
y_positions = np.arange(len(plot_order))
if len(SEEDS) == 1:
    single = results.set_index("model").loc[plot_order]
    accuracy_error = np.vstack([
        single.test_accuracy.to_numpy() - single.accuracy_ci_low.to_numpy(),
        single.accuracy_ci_high.to_numpy() - single.test_accuracy.to_numpy(),
    ])
    accuracy_upper = single.accuracy_ci_high.to_numpy()
    error_label = "95% image-level Wilson intervals"
else:
    accuracy_error = plot_summary.accuracy_std.to_numpy()
    accuracy_upper = np.minimum(1.0, (plot_summary.accuracy_mean + plot_summary.accuracy_std).to_numpy())
    error_label = "mean ± 1 seed SD"
fig, axes = plt.subplots(1, 2, figsize=(16, 7.5), sharey=True)
axes[0].barh(y_positions, plot_summary.accuracy_mean, xerr=accuracy_error,
             color=colors, alpha=0.9, capsize=3)
axes[1].barh(y_positions, plot_summary.macro_f1_mean, xerr=plot_summary.macro_f1_std,
             color=colors, alpha=0.9, capsize=3)
axes[0].set_yticks(y_positions, plot_labels)
axes[0].invert_yaxis()
for axis in axes:
    axis.set_xlim(0, 1.13)
    axis.set_xticks(np.linspace(0, 1, 6))
axes[0].set(title=f"Held-out test accuracy — {error_label}", xlabel="Accuracy")
axes[1].set(title="Held-out test macro F1 — mean ± seed SD", xlabel="Macro F1")
for index, row in enumerate(plot_summary.itertuples()):
    axes[0].text(min(accuracy_upper[index] + 0.015, 1.065), index,
                 f"{row.accuracy_mean:.1%}", va="center", fontsize=9)
    axes[1].text(min(row.macro_f1_mean + row.macro_f1_std + 0.015, 1.065), index,
                 f"{row.macro_f1_mean:.3f}", va="center", fontsize=9)
fig.suptitle(f"Brain Tumor MRI | mode: {RUN_MODE} | {len(SEEDS)} seed(s) | teal: quantum, indigo: classical", fontsize=13)
fig.tight_layout()
save_figure(fig, "06_test_accuracy_macro_f1.png")

### Cell 28 — Plot per-model training and validation accuracy

In [ ]:
PLOT_SEED = SEEDS[0]   # fixed in advance; not the best test seed
columns = 3
rows = int(np.ceil(len(plot_order) / columns))
fig, axes = plt.subplots(rows, columns, figsize=(16, 3.1 * rows), squeeze=False)
for axis, name in zip(axes.ravel(), plot_order):
    history = HISTORIES[(name, PLOT_SEED)]
    axis.plot(history.epoch, history.train_accuracy, marker=".", label="Training")
    axis.plot(history.epoch, history.val_accuracy, marker=".", label="Validation")
    axis.set(title=name.replace(" + auxiliary classifier", " + class head"),
             xlabel="Epoch", ylabel="Accuracy", ylim=(0, 1.05))
    axis.legend(fontsize=8)
for axis in axes.ravel()[len(plot_order):]:
    axis.axis("off")
fig.suptitle(f"Learning curves | fixed seed {PLOT_SEED}", fontsize=15)
fig.tight_layout()
save_figure(fig, "07_accuracy_learning_curves.png")

### Cell 29 — Plot classifier cross-entropy and QGAN adversarial losses

In [ ]:
fig, axes = plt.subplots(rows, columns, figsize=(16, 3.1 * rows), squeeze=False)
for axis, name in zip(axes.ravel(), plot_order):
    history = HISTORIES[(name, PLOT_SEED)]
    axis.plot(history.epoch, history.train_loss, label="Training (weighted CE)")
    axis.plot(history.epoch, history.val_loss, label="Validation (CE)")
    axis.set(title=name.replace(" + auxiliary classifier", " + class head"), xlabel="Epoch", ylabel="Loss")
    axis.legend(fontsize=8)
for axis in axes.ravel()[len(plot_order):]:
    axis.axis("off")
fig.tight_layout()
save_figure(fig, "08_classifier_loss_curves.png")

history = HISTORIES[("QGAN + auxiliary classifier", PLOT_SEED)]
fig, axis = plt.subplots(figsize=(9, 4))
axis.plot(history.epoch, history.discriminator_loss, label="Discriminator: adversarial + class")
axis.plot(history.epoch, history.generator_loss, label="Generator: adversarial + class")
axis.set(title="QGAN optimization losses (not accuracy)", xlabel="Epoch", ylabel="Loss")
axis.legend()
fig.tight_layout()
save_figure(fig, "09_qgan_optimization_losses.png")

### Cell 30 — Plot all held-out confusion matrices

In [ ]:
fig, axes = plt.subplots(rows, columns, figsize=(16, 3.8 * rows), squeeze=False)
for axis, name in zip(axes.ravel(), plot_order):
    prediction = PREDICTIONS[(name, PLOT_SEED)]
    matrix = confusion_matrix(prediction["truth"], prediction["predicted"], labels=range(N_CLASSES))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axis,
                vmin=0, vmax=int(torch.bincount(targets["test"], minlength=N_CLASSES).max()))
    axis.set(title=name.replace(" + auxiliary classifier", " + class head"),
             xlabel="Predicted class", ylabel="True class")
    axis.tick_params(axis="x", rotation=25)
    axis.tick_params(axis="y", rotation=0)
for axis in axes.ravel()[len(plot_order):]:
    axis.axis("off")
fig.suptitle(f"Held-out test confusion matrices | seed {PLOT_SEED}", fontsize=15)
fig.tight_layout()
save_figure(fig, "10_confusion_matrices.png")

### Cell 31 — Inspect a validation-selected classifier
Choose the model with the highest mean validation macro F1, then inspect the fixed
first seed's report, ROC curves, and misclassified MRI images. Test metrics do not
select the model or seed. ROC curves are descriptive one-vs-rest curves.

In [ ]:
selected_model_name = summary.sort_values(
    ["validation_macro_f1_mean", "validation_accuracy_mean"], ascending=False).iloc[0].model
selected_prediction = PREDICTIONS[(selected_model_name, PLOT_SEED)]
print("Selected using validation only:", selected_model_name, "| fixed seed:", PLOT_SEED)
print(classification_report(selected_prediction["truth"], selected_prediction["predicted"],
                            labels=range(N_CLASSES), target_names=CLASS_NAMES, zero_division=0))
fig, axis = plt.subplots(figsize=(7, 6))
for index, label in enumerate(CLASS_NAMES):
    binary_truth = (selected_prediction["truth"] == index).astype(int)
    fpr, tpr, _ = roc_curve(binary_truth, selected_prediction["probability"][:, index])
    axis.plot(fpr, tpr, label=f"{label}: AUC {auc(fpr, tpr):.3f}")
axis.plot([0, 1], [0, 1], linestyle="--", color="gray")
axis.set(xlabel="False positive rate", ylabel="True positive rate",
         title=f"One-vs-rest test ROC: {selected_model_name}")
axis.legend(loc="lower right")
fig.tight_layout()
save_figure(fig, "11_validation_selected_model_roc.png")

wrong = np.flatnonzero(selected_prediction["truth"] != selected_prediction["predicted"])
if len(wrong):
    inspected = wrong[:min(12, len(wrong))]
    error_rows = int(np.ceil(len(inspected) / 4))
    fig, axes = plt.subplots(error_rows, 4, figsize=(12, 3 * error_rows), squeeze=False)
    for axis, index in zip(axes.ravel(), inspected):
        axis.imshow(image_tensors["test"][index, 0], cmap="gray")
        truth_name = CLASS_NAMES[selected_prediction["truth"][index]]
        prediction_name = CLASS_NAMES[selected_prediction["predicted"][index]]
        confidence = selected_prediction["probability"][index].max()
        axis.set_title(f"True: {truth_name}\nPred: {prediction_name} ({confidence:.0%})", fontsize=9)
    for axis in axes.ravel():
        axis.axis("off")
    fig.tight_layout()
    save_figure(fig, "12_misclassified_images.png")
else:
    print("No misclassified test images for this run; inspect a larger independent test set before drawing conclusions.")

### Cell 32 — Inspect the QGAN's generated latent features
Compare the first two bounded PCA coordinates and per-class means/standard deviations
between real training features and generated features. These are generation diagnostics,
not MRI image quality, clinical validity, or a calibrated generative benchmark.

In [ ]:
seed_everything(PLOT_SEED + 1000)
generator = GAN_GENERATORS[PLOT_SEED].eval()
generated_by_class = {}
with torch.no_grad():
    for class_index, label in enumerate(CLASS_NAMES):
        generated_by_class[label] = generator(
            torch.randn(100, N_QUBITS), torch.full((100,), class_index, dtype=torch.long)).numpy()
fig, axes = plt.subplots(1, N_CLASSES, figsize=(15, 4), squeeze=False)
generation_rows = []
for class_index, label in enumerate(CLASS_NAMES):
    real = latent_features["train"][targets["train"] == class_index].numpy()
    generated = generated_by_class[label]
    axis = axes[0, class_index]
    axis.scatter(real[:300, 0], real[:300, 1], s=14, alpha=0.4, label="Real training")
    axis.scatter(generated[:, 0], generated[:, 1], s=14, alpha=0.5, label="Generated")
    axis.set(title=label, xlabel="Bounded PCA coordinate 1", ylabel="Bounded PCA coordinate 2",
             xlim=(-1.05, 1.05), ylim=(-1.05, 1.05))
    generation_rows.append({"class": label,
        "mean_absolute_mean_gap": float(np.abs(real.mean(0) - generated.mean(0)).mean()),
        "mean_absolute_std_gap": float(np.abs(real.std(0) - generated.std(0)).mean()),
        "n_generated": len(generated)})
axes[0, 0].legend(fontsize=8)
fig.suptitle("QGAN latent generation: real training features vs generated features")
fig.tight_layout()
save_figure(fig, "13_qgan_latent_diagnostics.png")
generation_diagnostics = pd.DataFrame(generation_rows)
display(generation_diagnostics.round(4))
generation_diagnostics.to_csv(RUN_DIR / "qgan_latent_diagnostics.csv", index=False)
np.savez(RUN_DIR / f"qgan_generated_latents_seed_{PLOT_SEED}.npz", **generated_by_class)

### Cell 33 — Compare quantum models with their corresponding controls
These differences compare mean results across the same seeds and sample identities.
They are exploratory effect sizes, not significance tests. They do not control every
parameter count or optimization choice. VQC uses a separate learning rate for its
probability readout; stronger conclusions require equally budgeted tuning on validation.

In [ ]:
matched_pairs = [
    ("Quanvolution", "Patch CNN control"),
    ("Hybrid quantum CNN", "CNN bottleneck control"),
    ("VQC", "Latent MLP control"),
    ("QGAN + auxiliary classifier", "Latent MLP control"),
    ("Angle QNN", "Latent MLP control"),
    ("Amplitude QNN", "Amplitude MLP control"),
]
indexed_summary = summary.set_index("model")
pair_rows = []
for quantum_name, control_name in matched_pairs:
    quantum_row, control_row = indexed_summary.loc[quantum_name], indexed_summary.loc[control_name]
    pair_rows.append({"quantum_model": quantum_name, "classical_control": control_name,
                      "accuracy_difference_pp": 100 * (quantum_row.accuracy_mean - control_row.accuracy_mean),
                      "macro_f1_difference": quantum_row.macro_f1_mean - control_row.macro_f1_mean})
matched_comparison = pd.DataFrame(pair_rows)
display(matched_comparison.round(4))
matched_comparison.to_csv(RUN_DIR / "matched_control_comparison.csv", index=False)

## Checks and export
### Cell 34 — Check outputs and save reproducibility information
Keep the raw, quarantined, and final manifests with your results. The feature-transform
NPZ contains PCA components and training-only scaler parameters; the fixed quanvolution
kernel is saved separately. Checkpoints contain only state dictionaries and configuration.
For loading your own saved checkpoint, construct its model class, call
`torch.load(path, map_location="cpu", weights_only=True)`, then `load_state_dict`.
Feature extraction must use the saved training transforms, not refit PCA on new images.

In [ ]:
assert all(len(prediction["truth"]) == len(test_manifest) for prediction in PREDICTIONS.values())
assert all(np.isfinite(prediction["probability"]).all() for prediction in PREDICTIONS.values())
assert all(np.allclose(prediction["probability"].sum(axis=1), 1, atol=1e-5)
           for prediction in PREDICTIONS.values())
assert len(list((RUN_DIR / "histories").glob("*.csv"))) == len(expected_names) * len(SEEDS)
assert (RUN_DIR / "plots" / "06_test_accuracy_macro_f1.png").exists()
versions = {package: metadata.version(package) for package in
            ["torch", "numpy", "scipy", "pandas", "scikit-learn", "Pillow",
             "matplotlib", "seaborn", "PennyLane", "PennyLane-Lightning", "autoray", "kagglehub"]}
environment = {"python": platform.python_version(), "packages": versions,
               "cuda_available": torch.cuda.is_available(),
               "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
               "classical_device": str(CLASSICAL_DEVICE), "quantum_device": str(QUANTUM_DEVICE)}
(RUN_DIR / "environment.json").write_text(json.dumps(environment, indent=2))
(RUN_DIR / "requirements_observed.txt").write_text("\n".join(f"{name}=={version}" for name, version in versions.items()) + "\n")
run_notes = {
    "run_mode": RUN_MODE, "full_dataset_run": RUN_MODE == "full", "seeds": SEEDS,
    "validation_selected_model": selected_model_name, "plot_seed": PLOT_SEED,
    "patient_independence_verified": False,
    "split_type": "image-level, deduplicated official test with stratified validation from official train",
    "exact_duplicates_checked": True, "possible_near_duplicate_groups": len(near_cross_split),
    "near_duplicates_automatically_removed": False,
    "quantum_execution": "CPU statevector simulation, analytic expectation values, backpropagation",
    "qgan_accuracy_definition": "auxiliary tumor-class head accuracy on real held-out MRI features",
    "qgan_generated_output": "bounded PCA latent vectors, not MRI images",
    "n_train": len(train_manifest), "n_val": len(val_manifest), "n_test": len(test_manifest),
    "dataset_identity_sha256": hashlib.sha256(
        "\n".join(sorted(raw_manifest.relative_path + ":" + raw_manifest.sha256)).encode()
    ).hexdigest(),
}
(RUN_DIR / "run_notes.json").write_text(json.dumps(run_notes, indent=2))
print("PASS: all 11 models recorded, prediction probabilities valid, histories and plots saved.")
print("Actual output directory:", RUN_DIR)

### Cell 35 — Package and optionally download the results
The archive contains metrics, plots, histories, checkpoints, manifests, transforms,
and generated latent features. The downloaded source dataset is excluded.
Set `DOWNLOAD_RESULTS=True` to trigger a Colab download after the ZIP is created.

In [ ]:
DOWNLOAD_RESULTS = False
archive_path = RUN_DIR.parent / (RUN_DIR.name + "_results.zip")
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file() and "downloaded_dataset" not in path.relative_to(RUN_DIR).parts:
            archive.write(path, arcname=str(Path(RUN_DIR.name) / path.relative_to(RUN_DIR)))
print("Results archive:", archive_path)
print("Archive size:", round(archive_path.stat().st_size / 1024**2, 2), "MiB")
if DOWNLOAD_RESULTS:
    if COLAB:
        from google.colab import files
        files.download(str(archive_path))
    else:
        print("Outside Colab, retrieve the archive from the printed path.")

## Next steps — Interpreting your results
- **Report measured values**, the exact retained image counts, preprocessing, seed count,
  and the deduplicated test protocol. Do not compare them directly to papers that use a
  different split or retain duplicate images.
- **Compare the matched controls first.** Compression can reduce accuracy independently
  of the quantum circuit. The 64×64 CNN and 6-feature VQC do not have equal information.
- **Increase SEEDS** for repeated results. Validation selects checkpoints; the test set
  does not choose hyperparameters, training fractions, qubit counts, or a best seed.
- **Review near duplicates** and obtain patient identifiers before claiming a
  patient-independent medical imaging result. Use grouped splits and external data
  when those identities and sources are available.
- **For QGAN**, real-image class accuracy and latent moment gaps do not establish
  generative quality. Compare against a conditional classical GAN before attributing
  a benefit to the quantum generator. Check for mode collapse and class overlap.
- **For a research study**, add a strong pretrained image baseline, equally budgeted
  hyperparameter tuning, circuit/encoding ablations, measured resource costs, finite-shot
  and noise experiments, and paired uncertainty analysis on a valid independent unit.

A high accuracy by itself does not establish novelty or publication readiness. This
notebook provides a reproducible experimental starting point; it does not prescribe
the outcome or promise a quantum advantage.

### References and implementation sources
1. [Dataset, version 1](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/1).
2. Henderson et al., [Quanvolutional Neural Networks](https://arxiv.org/abs/1904.04767).
3. Cong et al., [Quantum Convolutional Neural Networks](https://arxiv.org/abs/1810.03787).
4. [PennyLane quanvolution tutorial](https://pennylane.ai/demos/tutorial_quanvolution).
5. [PennyLane PyTorch interface](https://docs.pennylane.ai/en/stable/introduction/interfaces/torch.html).
6. [AngleEmbedding](https://docs.pennylane.ai/en/stable/code/api/pennylane.AngleEmbedding.html)
   and [AmplitudeEmbedding](https://docs.pennylane.ai/en/stable/code/api/pennylane.AmplitudeEmbedding.html).
7. Zoufal et al., [Quantum GANs for Learning and Loading Random Distributions](https://arxiv.org/abs/1904.00043).
8. [PennyLane quantum GAN tutorial](https://pennylane.ai/demos/tutorial_quantum_gans).
9. [KaggleHub official repository](https://github.com/Kaggle/kagglehub).

The circuits and models here are compact experimental implementations inspired by
these methods, not claims to reproduce the cited papers' results or architectures.